# TCREmp 10x dcode Analysis

This notebook reproduces the core 10x dcode workflow with the new CITE-seq infrastructure:
- load donor-level paired VDJ plus binarized CITE-seq matrix into `SingleCellSample`;
- sanity-check binder definitions against VDJdb records with 10x references;
- derive donor1 pseudo-labels from positive binder channels;
- run paired TCREmp embedding and report DBSCAN purity/consistency diagnostics.

In [1]:
# Set deterministic seed and print environment versions used in this analysis.
from __future__ import annotations

import random
import time
from pathlib import Path

import numpy as np
import polars as pl
import sklearn

from mir.common.single_cell import (
    load_10x_vdj_v1_citeseq_sample,
    validate_citeseq_binders_against_vdjdb_10x,
)
from mir.embedding.tcremp import PairedTCREmp
from mir.utils.embedding_diagnostics import analyze_embedding_dbscan
from mir.utils.notebook_assets import (
    ensure_airr_benchmark,
    find_airr_benchmark_dcode_10x_vdj_v1_donor,
    find_airr_benchmark_dcode_10x_vdj_v1_donor_matrix,
    find_airr_benchmark_vdjdb_full,
    find_repo_root,
)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print(f'Python: {__import__("sys").version.split()[0]}')
print(f'numpy: {np.__version__}')
print(f'polars: {pl.__version__}')
print(f'scikit-learn: {sklearn.__version__}')

/Users/mikesh/vcs/mirpy/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.12
numpy: 2.4.4
polars: 1.39.3
scikit-learn: 1.8.0


In [9]:
# Load AIRR benchmark assets and build SingleCellSample objects for all 10x_vdj_v1 donors.
repo_root = find_repo_root(Path.cwd())
dataset_root = ensure_airr_benchmark(repo_root=repo_root, allow_patterns=['dcode/*', 'vdjdb/**'])
vdjdb_full = find_airr_benchmark_vdjdb_full(dataset_root)

donor_ids = ['donor1', 'donor2', 'donor3', 'donor4']
samples = {}
donor_paths = {}

for donor_id in donor_ids:
    all_contig, consensus = find_airr_benchmark_dcode_10x_vdj_v1_donor(dataset_root, donor_id)
    matrix = find_airr_benchmark_dcode_10x_vdj_v1_donor_matrix(dataset_root, donor_id)
    donor_paths[donor_id] = {'all_contig': all_contig, 'consensus': consensus, 'matrix': matrix}
    t0 = time.perf_counter()
    sample = load_10x_vdj_v1_citeseq_sample(
        consensus_annotations_path=consensus,
        all_contig_annotations_path=all_contig,
        binarized_matrix_path=matrix,
        sample_id=donor_id,
    )
    dt = time.perf_counter() - t0
    samples[donor_id] = sample
    print(
        f'{donor_id}: cells={sample.paired_repertoire.loaded_cell_count:,} '
        f'pairs={sample.paired_repertoire.paired_locus_repertoires["TRA_TRB"].clonotype_count:,} '
        f'matrix_rows={sample.cite_seq_matrix.height:,} binder_cols={sample.cite_seq_binder_columns.height:,} '
        f'time={dt:.2f}s'
    )

Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 2031.40it/s]


donor1: cells=47,271 pairs=48,890 matrix_rows=46,526 binder_cols=78 time=1.30s
donor2: cells=79,704 pairs=72,266 matrix_rows=77,854 binder_cols=78 time=1.76s
donor3: cells=38,095 pairs=39,518 matrix_rows=37,824 binder_cols=78 time=1.16s
donor4: cells=27,640 pairs=29,147 matrix_rows=27,308 binder_cols=78 time=0.95s


In [10]:
# Validate donor binder definitions against VDJdb records annotated with 10x references.
sanity_rows = []
for donor_id, sample in samples.items():
    missing = validate_citeseq_binders_against_vdjdb_10x(sample.cite_seq_binder_columns, vdjdb_full)
    sanity_rows.append({
        'donor_id': donor_id,
        'binder_columns': sample.cite_seq_binder_columns.height,
        'missing_targets_vs_vdjdb_10x': missing.height,
    })
    if missing.height > 0:
        print(f'\n{donor_id} residual unmatched targets:')
        print(missing)

sanity_df = pl.DataFrame(sanity_rows).sort('donor_id')
print(sanity_df)


donor1 residual unmatched targets:
shape: (2, 5)
┌─────────────────────────────────┬───────┬─────────────────┬──────────────┬─────────────────┐
│ column                          ┆ hla   ┆ antigen.epitope ┆ antigen.gene ┆ antigen.species │
│ ---                             ┆ ---   ┆ ---             ┆ ---          ┆ ---             │
│ str                             ┆ str   ┆ str             ┆ str          ┆ str             │
╞═════════════════════════════════╪═══════╪═════════════════╪══════════════╪═════════════════╡
│ A0201_CLGGLLTMV_LMP-2A_EBV      ┆ A0201 ┆ CLGGLLTMV       ┆ LMP-2A       ┆ EBV             │
│ A0201_LLMGTLGIVC_HPV-16E7_82-9… ┆ A0201 ┆ LLMGTLGIVC      ┆ HPV-16E7     ┆ 82-91           │
└─────────────────────────────────┴───────┴─────────────────┴──────────────┴─────────────────┘

donor2 residual unmatched targets:
shape: (2, 5)
┌─────────────────────────────────┬───────┬─────────────────┬──────────────┬─────────────────┐
│ column                          ┆ hla   ┆ a

In [11]:
# Build donor1 paired clonotype labels from positive CITE-seq binder channels and prepare embeddings.
def donor1_binder_labels(sample):
    binder_cols = sample.cite_seq_binder_columns.filter(pl.col('is_binder') == '1').get_column('column').to_list()
    if not binder_cols:
        raise ValueError('No binder columns found in donor CITE-seq matrix')

    long = (
        sample.cite_seq_matrix
        .select(['barcode', *binder_cols])
        .unpivot(index='barcode', variable_name='column', value_name='value')
        .filter(pl.col('value') == 1)
    )
    if long.height == 0:
        raise ValueError('No positive binder events found')

    info = sample.cite_seq_binder_columns.select(['column', 'hla', 'antigen.epitope'])
    labels = (
        long.join(info, on='column', how='left')
        .group_by('barcode', 'antigen.epitope')
        .len()
        .sort(['barcode', 'len', 'antigen.epitope'], descending=[False, True, False])
        .group_by('barcode')
        .first()
        .select(['barcode', pl.col('antigen.epitope').alias('epitope_label')])
    )
    return labels


donor1 = samples['donor1']
labels_df = donor1_binder_labels(donor1)

# Link CITE barcodes to clonotype IDs using all-contig annotations.
all_contig_df = pl.read_csv(donor_paths['donor1']['all_contig']).select(['barcode', 'raw_clonotype_id']).drop_nulls()
clonotype_labels = (
    all_contig_df.join(labels_df, on='barcode', how='inner')
    .group_by('raw_clonotype_id', 'epitope_label')
    .len()
    .sort(['raw_clonotype_id', 'len', 'epitope_label'], descending=[False, True, False])
    .group_by('raw_clonotype_id')
    .first()
    .rename({'raw_clonotype_id': 'clonotype_id'})
    .select(['clonotype_id', 'epitope_label'])
)
clonotype_label_map = {
    row['clonotype_id']: row['epitope_label']
    for row in clonotype_labels.iter_rows(named=True)
}

pairs = []
rows = []
for pair in donor1.paired_repertoire.paired_locus_repertoires['TRA_TRB'].paired_clonotypes:
    clonotype_id = pair.pair_id.split('_', 1)[0]
    epitope = clonotype_label_map.get(clonotype_id)
    if epitope is None:
        continue
    pairs.append(pair)
    rows.append({'clonotype_id': clonotype_id, 'epitope_label': epitope})

label_table = pl.DataFrame(rows)
label_counts = label_table.group_by('epitope_label').len().sort('len', descending=True)

print(f'Paired clonotypes with donor1 CITE labels: {len(pairs):,}')
print(label_counts.head(10))

Paired clonotypes with donor1 CITE labels: 22,035
shape: (10, 2)
┌───────────────┬───────┐
│ epitope_label ┆ len   │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ KLGGALQAK     ┆ 10598 │
│ AVFDRKSDAK    ┆ 3799  │
│ IVTDFSVIK     ┆ 3450  │
│ GILGFVFTL     ┆ 2858  │
│ RAKFKQLL      ┆ 599   │
│ ELAGIGILTV    ┆ 276   │
│ LLDFVRFMGV    ┆ 158   │
│ RLRAEAQVK     ┆ 105   │
│ GLCTLVAML     ┆ 63    │
│ FLYALALLL     ┆ 23    │
└───────────────┴───────┘


In [12]:
# Embed donor1 paired clonotypes and report clustering quality with shared diagnostics helpers.
MAX_PER_EPITOPE = 250
OTHER_LABEL = 'other'
TOP_LABELS = label_table.group_by('epitope_label').len().sort('len', descending=True).head(10).get_column('epitope_label').to_list()

balanced_parts = []
for label in TOP_LABELS:
    part = label_table.filter(pl.col('epitope_label') == label)
    balanced_parts.append(part.sample(n=min(MAX_PER_EPITOPE, part.height), seed=SEED, shuffle=True))

other = label_table.filter(~pl.col('epitope_label').is_in(TOP_LABELS))
if other.height > 0:
    balanced_parts.append(other.sample(n=min(500, other.height), seed=SEED, shuffle=True).with_columns(pl.lit(OTHER_LABEL).alias('epitope_label')))

balanced = pl.concat(balanced_parts, how='vertical')
pair_index = {pair.pair_id.split('_', 1)[0]: pair for pair in pairs}
balanced_pairs = [pair_index[c] for c in balanced.get_column('clonotype_id').to_list()]
balanced_labels = balanced.get_column('epitope_label').to_numpy()

model = PairedTCREmp.from_defaults(species='human', locus_pair='TRA_TRB', n_prototypes=500, junction_method='fixed_gap')
t0 = time.perf_counter()
X = model.embed(balanced_pairs)
embed_time = time.perf_counter() - t0
analysis = analyze_embedding_dbscan(X, balanced_labels, seed=SEED)

print(f'Balanced donor1 records: {balanced.height:,}')
print(f'Embedding runtime: {embed_time:.3f}s ({1e3 * embed_time / max(balanced.height, 1):.3f} ms/record)')
print({
    'pcs_90pct': analysis['n_comp'],
    'eps': analysis['eps'],
    'clusters': analysis['n_clusters'],
    'retention': analysis['retention'],
    'purity': analysis['purity'],
    'consistency_70': analysis['consistency'],
    'median_4nn': analysis['median_4nn'],
})

Balanced donor1 records: 1,955
Embedding runtime: 0.058s (0.030 ms/record)
{'pcs_90pct': 39, 'eps': 1e-06, 'clusters': 100, 'retention': 0.5202046035805626, 'purity': 0.975, 'consistency_70': 0.95, 'median_4nn': 0.3111549913883209}
